# Phase 2: YOLOv8n Training Notebook — Diagram-to-Infra-Code

This notebook trains a custom **YOLOv8n** object detection model on synthetic architecture diagram data.

### Target Classes (7 total):
1. `compute` (EC2, EKS)
2. `database` (RDS)
3. `storage` (S3)
4. `load_balancer` (ALB)
5. `network` (VPC/Subnet boundary)
6. `arrow` (directional connection)
7. `text_label` (label box text)

Run this notebook on Google Colab with GPU acceleration enabled (**Runtime -> Change runtime type -> T4 GPU**).

## Step 1: GPU Check & Environment Setup

In [ ]:
!nvidia-smi
!pip install -q ultralytics huggingface_hub opencv-python-headless albumentations pillow

## Step 2: Clone Repository & Generate Dataset

In [ ]:
!git clone https://github.com/Parths-29/Diagram-to-Infra-code.git
%cd Diagram-to-Infra-code

# Generate 500 unique diagrams + 1 copy = 1,000 total images
!python3 data/generate_synthetic.py --output data/synthetic_dataset --count 500 --augmented-copies 1 --seed 42

## Step 3: Run YOLOv8n Training

Explicit augmentation kwargs:
- `fliplr=0.0` and `flipud=0.0` (disabled horizontal/vertical flips to preserve text & arrow orientation)
- `degrees=10.0`, `translate=0.1`, `scale=0.2`, `mosaic=0.5`, `mixup=0.1`

In [ ]:
!python3 ml/train.py --data data/dataset.yaml --epochs 50 --batch 16 --imgsz 640 --weights-out ml/weights --eval-out ml/eval_samples

## Step 4: Display Training Results & Visual Evaluation Samples

In [ ]:
from IPython.display import Image, display
import glob

print("=== Training Confusion Matrix ===")
display(Image(filename="runs/detect/diagram_yolov8n/confusion_matrix.png"))

print("=== Results Curves ===")
display(Image(filename="runs/detect/diagram_yolov8n/results.png"))

print("=== Sample Visual Predictions (from ml/eval_samples/) ===")
eval_images = glob.glob("ml/eval_samples/*.png")[:5]
for img_path in eval_images:
    display(Image(filename=img_path))

## Step 5: (Optional) Push Trained Weights to Hugging Face Hub

In [ ]:
import os
from huggingface_hub import HfApi

# Set your HF_TOKEN here if pushing to HF Hub:
HF_TOKEN = ""  # e.g., "hf_..."
REPO_ID = "parths-29/diagram-to-infra-yolov8n"

if HF_TOKEN:
    api = HfApi()
    api.create_repo(repo_id=REPO_ID, exist_ok=True, token=HF_TOKEN)
    api.upload_file(
        path_or_fileobj="ml/weights/best.pt",
        path_in_repo="best.pt",
        repo_id=REPO_ID,
        token=HF_TOKEN
    )
    print(f"Uploaded best.pt to https://huggingface.co/{REPO_ID}")
else:
    print("No HF_TOKEN provided. Weights are saved locally at ml/weights/best.pt")